# 中医中药多标签预测模型

**任务**：根据「症状规范 + YOLO标签」预测「中药规范」（多标签分类）

**特征**：症状规范（前200高频症状）+ YOLO标签（共6种）

**标签**：取频次前150的中药

**模型**：Logistic Regression（OneVsRest，与证型模型保持一致）

**流程**：先用 80/20 分割评估指标 → 再用全量数据重训练保存最终模型

## 1. 导入依赖

In [13]:
import pandas as pd
import numpy as np
import json
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from collections import Counter
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    hamming_loss, jaccard_score,
    f1_score, accuracy_score, roc_auc_score,
    classification_report
)

print('依赖导入完成')

依赖导入完成


## 2. 数据加载与清洗

In [14]:
DATA_PATH = 'data_final.csv'

df = pd.read_csv(DATA_PATH)
print(f'原始数据形状: {df.shape}')
print(f'列名: {df.columns.tolist()}')

# 过滤：只保留「中药规范」非空的行
df = df[df['中药规范'].notna() & (df['中药规范'].str.strip() != '')].reset_index(drop=True)
print(f'过滤空中药后: {df.shape}')

def parse_list(s):
    if pd.isna(s) or str(s).strip() == '':
        return []
    return [x.strip() for x in str(s).split(',') if x.strip()]

df.head(3)

原始数据形状: (1562, 5)
列名: ['症状规范', '中药规范', '证型规范', '舌象', 'YOLO标签']
过滤空中药后: (1562, 5)


,症状规范,中药规范,证型规范,舌象,YOLO标签
0,"咳嗽,喘息,头晕,汗出","人参,鹿茸,红景天,苍术,厚朴,李子,车前子,白附子,法半夏,紫菀,莪术,款冬花,金荞麦,冬...","脾肾亏虚,痰热壅肺","舌质暗,苔薄黄腻",red tongue yellow fur thick greasy fur
1,"下肢浮肿,汗出,纳差,面色晦暗","红景天,苍术,厚朴,李子,车前子,白附子,法半夏,紫菀,莪术,款冬花,金荞麦,冬瓜子,忍冬藤...","脾肾亏虚,痰热壅肺","舌质暗,苔薄白",The white tongue is thick and greasy
2,"头晕,纳差","人参,鹿茸,红景天,莱菔子,芥子,厚朴,大皂角,当归,莪术,紫菀,款冬花,法半夏,蚕沙,桑白...","脾肾亏虚,痰热壅肺",NaN,NaN


## 3. 确定特征集与标签集（基于全量数据统计）

> 统计频次必须用**全量数据**，保证前150中药 / 前200症状的定义与实际数据一致。

In [15]:
TOP_N_SYMPTOMS = 200
TOP_K_HERBS    = 150

# ── 前150中药 ──
herb_counter = Counter()
for ls in df['中药规范'].apply(parse_list): herb_counter.update(ls)
top_herbs = [h for h, _ in herb_counter.most_common(TOP_K_HERBS)]
print(f'中药总种类: {len(herb_counter)}，取前 {TOP_K_HERBS}')
print(f'\n前 10 高频中药:')
for h, c in herb_counter.most_common(10): print(f'  {h}: {c} 次')
print(f'\n第{TOP_K_HERBS}名中药: {top_herbs[-1]}（频次: {herb_counter[top_herbs[-1]]}）')

# ── 前200症状 ──
sx_counter = Counter()
for ls in df['症状规范'].apply(parse_list): sx_counter.update(ls)
top_symptoms = [s for s, _ in sx_counter.most_common(TOP_N_SYMPTOMS)]
print(f'\n症状总种类: {len(sx_counter)}，取前 {TOP_N_SYMPTOMS}')
print(f'第{TOP_N_SYMPTOMS}名症状: {top_symptoms[-1]}（频次: {sx_counter[top_symptoms[-1]]}）')

# ── YOLO 标签（全部保留）──
yolo_counter = Counter()
for y in df['YOLO标签'].dropna():
    y = str(y).strip()
    if y: yolo_counter[y] += 1
all_yolo_labels = sorted(yolo_counter.keys())
print(f'\nYOLO标签共 {len(all_yolo_labels)} 种: {all_yolo_labels}')

中药总种类: 607，取前 150

前 10 高频中药:
  茯苓: 746 次
  苦杏仁: 639 次
  黄芪: 637 次
  紫苏子: 603 次
  甘草: 603 次
  白术: 580 次
  五味子: 549 次
  陈皮: 542 次
  麻黄: 512 次
  黄芩: 440 次

第150名中药: 紫苏梗（频次: 29）

症状总种类: 433，取前 200
第200名症状: 迂曲（频次: 1）

YOLO标签共 6 种: ['The red tongue is thick and greasy', 'The white tongue is thick and greasy', 'black tongue coating', 'map tongue coating', 'purple tongue coating', 'red tongue yellow fur thick greasy fur']


## 4. 构建全量特征矩阵与标签矩阵

In [16]:
# 过滤：只保留至少含1个top-150中药的样本
top_herb_set = set(top_herbs)
df['herbs_filtered'] = df['中药规范'].apply(
    lambda s: [h for h in parse_list(s) if h in top_herb_set]
)
df_valid = df[df['herbs_filtered'].apply(len) > 0].reset_index(drop=True)
print(f'有效样本数（含top-{TOP_K_HERBS}中药）: {len(df_valid)}')

# 症状特征：固定 classes=top_symptoms，维度始终为 200
mlb_symptom = MultiLabelBinarizer(classes=top_symptoms)
X_sym = mlb_symptom.fit_transform(
    df_valid['症状规范'].apply(lambda s: [x for x in parse_list(s) if x in set(top_symptoms)])
)

# YOLO 特征：固定 classes=all_yolo_labels，维度始终为 6
mlb_yolo = MultiLabelBinarizer(classes=all_yolo_labels)
X_yolo = mlb_yolo.fit_transform(
    df_valid['YOLO标签'].apply(lambda y: [str(y).strip()] if pd.notna(y) and str(y).strip() else [])
)

# 合并：(N, 206)
X_all = np.hstack([X_sym, X_yolo])
print(f'特征矩阵: {X_all.shape}  (前{TOP_N_SYMPTOMS}症状 + {len(all_yolo_labels)} YOLO标签)')

# 标签矩阵（前150中药）
mlb_herb = MultiLabelBinarizer(classes=top_herbs)
Y_all = mlb_herb.fit_transform(df_valid['herbs_filtered'])
print(f'标签矩阵: {Y_all.shape}  (前{TOP_K_HERBS}中药)')
print(f'\n平均每条样本标注中药数: {Y_all.sum(axis=1).mean():.1f}')

有效样本数（含top-150中药）: 1562
特征矩阵: (1562, 206)  (前200症状 + 6 YOLO标签)
标签矩阵: (1562, 150)  (前150中药)

平均每条样本标注中药数: 14.2


## 5. 划分训练集 / 测试集（用于评估）

In [17]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X_all, Y_all, test_size=0.2, random_state=42
)
print(f'训练集: {X_train.shape}, 测试集: {X_test.shape}')

训练集: (1249, 206), 测试集: (313, 206)


## 6. 评估阶段：在 train/test 分割上训练并评估

In [18]:
def make_model():
    """返回一个新的、未训练的模型实例"""
    return OneVsRestClassifier(
        LogisticRegression(
            C=0.1,
            max_iter=1000,
            class_weight='balanced',  # 解决标签不平衡
            solver='lbfgs',
            random_state=42
        ),
        n_jobs=-1
    )

print('[1/2] 在 train/test 分割上训练（用于评估）...')
eval_model = make_model()
eval_model.fit(X_train, Y_train)
print('完成！')

[1/2] 在 train/test 分割上训练（用于评估）...
完成！


## 7. 模型评估（在 20% 测试集上）

In [19]:
Y_pred       = eval_model.predict(X_test)
Y_pred_proba = eval_model.predict_proba(X_test)

hl   = hamming_loss(Y_test, Y_pred)
acc  = accuracy_score(Y_test, Y_pred)
jacc = jaccard_score(Y_test, Y_pred, average='samples', zero_division=0)
f1mi = f1_score(Y_test, Y_pred, average='micro', zero_division=0)
f1ma = f1_score(Y_test, Y_pred, average='macro', zero_division=0)

try:
    auc_macro = roc_auc_score(Y_test, Y_pred_proba, average='macro')
except Exception as e:
    auc_macro = float('nan')
    print(f'[警告] Macro AUC 计算失败: {e}')

sep = '=' * 52
print(sep)
print('评估结果 — 中药预测 Logistic Regression (80%训练/20%测试)')
print(sep)
print(f'  Hamming Loss      : {hl:.4f}   ↓越低越好')
print(f'  Subset Accuracy   : {acc:.4f}   ↑越高越好')
print(f'  Jaccard Score     : {jacc:.4f}   ↑越高越好')
print(f'  F1-Score (Micro)  : {f1mi:.4f}   ↑越高越好')
print(f'  F1-Score (Macro)  : {f1ma:.4f}   ↑越高越好')
print(f'  Macro AUC         : {auc_macro:.4f}   ↑越高越好')
print(sep)
print('\n各中药详细分类报告（前150）:')
print(classification_report(Y_test, Y_pred, target_names=mlb_herb.classes_, zero_division=0))

评估结果 — 中药预测 Logistic Regression (80%训练/20%测试)
  Hamming Loss      : 0.3085   ↓越低越好
  Subset Accuracy   : 0.0000   ↑越高越好
  Jaccard Score     : 0.1386   ↑越高越好
  F1-Score (Micro)  : 0.2391   ↑越高越好
  F1-Score (Macro)  : 0.1882   ↑越高越好
  Macro AUC         : 0.6344   ↑越高越好

各中药详细分类报告（前150）:
              precision    recall  f1-score   support

          茯苓       0.56      0.62      0.59       150
         苦杏仁       0.54      0.57      0.55       124
          黄芪       0.53      0.69      0.60       127
         紫苏子       0.46      0.56      0.50       109
          甘草       0.43      0.58      0.50       118
          白术       0.45      0.60      0.52       115
         五味子       0.42      0.57      0.48       104
          陈皮       0.39      0.53      0.45        98
          麻黄       0.43      0.50      0.46       104
          黄芩       0.37      0.58      0.45        85
         法半夏       0.32      0.49      0.39        83
         炙甘草       0.24      0.45      0.32        71
          紫

## 8. 全量重训练并保存最终模型

> **标准实践**：评估后用**全量数据**再训练一遍，模型见过更多样本，泛化能力更强。
> 保存的评估指标来自上一步的 hold-out 测试集，是对真实性能的客观估计。

In [20]:
# ── [2/2] 全量数据重训练 ──
print('[2/2] 用全量数据重训练最终模型（更完整）...')
final_model = make_model()
final_model.fit(X_all, Y_all)   # ← 全量，比评估阶段多 20% 样本
print(f'    训练样本数: {len(X_all)}（比评估阶段多 {len(X_all)-len(X_train)} 条）')
print('    重训练完成！')

# ── 保存 ──
SAVE_DIR = 'tcm_herb_model'
os.makedirs(SAVE_DIR, exist_ok=True)

with open(f'{SAVE_DIR}/model.pkl',       'wb') as f: pickle.dump(final_model, f)
with open(f'{SAVE_DIR}/mlb_symptom.pkl', 'wb') as f: pickle.dump(mlb_symptom, f)
with open(f'{SAVE_DIR}/mlb_yolo.pkl',   'wb') as f: pickle.dump(mlb_yolo,    f)
with open(f'{SAVE_DIR}/mlb_herb.pkl',   'wb') as f: pickle.dump(mlb_herb,    f)

with open(f'{SAVE_DIR}/top_symptoms.json', 'w', encoding='utf-8') as f:
    json.dump(top_symptoms,  f, ensure_ascii=False, indent=2)
with open(f'{SAVE_DIR}/top_herbs.json', 'w', encoding='utf-8') as f:
    json.dump(top_herbs,     f, ensure_ascii=False, indent=2)
with open(f'{SAVE_DIR}/yolo_labels.json', 'w', encoding='utf-8') as f:
    json.dump(all_yolo_labels, f, ensure_ascii=False, indent=2)

meta = {
    'model_type': 'LogisticRegression (OneVsRest, C=0.1, class_weight=balanced)',
    'task': 'herb_prediction',
    'trained_on': 'full dataset',
    'n_samples_full': int(len(X_all)),
    'n_samples_eval_train': int(len(X_train)),
    'n_samples_eval_test': int(len(X_test)),
    'top_n_symptoms': TOP_N_SYMPTOMS,
    'top_k_herbs': TOP_K_HERBS,
    'n_features': int(X_all.shape[1]),
    'n_labels': int(Y_all.shape[1]),
    'symptom_list': top_symptoms,
    'herb_list': top_herbs,
    'yolo_label_list': all_yolo_labels,
    'eval_metrics': {
        'note': '以下指标来自 80/20 hold-out 评估，非全量训练集',
        'hamming_loss': round(hl, 4),
        'subset_accuracy': round(acc, 4),
        'jaccard': round(jacc, 4),
        'f1_micro': round(f1mi, 4),
        'f1_macro': round(f1ma, 4),
        'macro_auc': round(auc_macro, 4) if not np.isnan(auc_macro) else None
    }
}
with open(f'{SAVE_DIR}/meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f'\n✅ 所有文件已保存到: {os.path.abspath(SAVE_DIR)}')
for fname in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(f'{SAVE_DIR}/{fname}')
    print(f'  {fname:30s}  {size/1024:.1f} KB')

[2/2] 用全量数据重训练最终模型（更完整）...
    训练样本数: 1562（比评估阶段多 313 条）
    重训练完成！

✅ 所有文件已保存到: /Users/yangchenghao/PycharmProjects/graduation_project/machinelearning/tcm_herb_model
  meta.json                       6.5 KB
  mlb_herb.pkl                    2.7 KB
  mlb_symptom.pkl                 3.9 KB
  mlb_yolo.pkl                    0.5 KB
  model.pkl                       302.9 KB
  top_herbs.json                  1.9 KB
  top_symptoms.json               3.1 KB
  yolo_labels.json                0.2 KB


## 9. Django 调用示例

In [21]:
# ── 1. 加载模型（Django 应用启动时执行一次）──
import pickle, json, numpy as np

SAVE_DIR = 'tcm_herb_model'

with open(f'{SAVE_DIR}/model.pkl',       'rb') as f: _model       = pickle.load(f)
with open(f'{SAVE_DIR}/mlb_symptom.pkl', 'rb') as f: _mlb_symptom = pickle.load(f)
with open(f'{SAVE_DIR}/mlb_yolo.pkl',   'rb') as f: _mlb_yolo    = pickle.load(f)
with open(f'{SAVE_DIR}/mlb_herb.pkl',   'rb') as f: _mlb_herb    = pickle.load(f)
with open(f'{SAVE_DIR}/top_symptoms.json', encoding='utf-8') as f: _symptom_list = json.load(f)

# ── 2. 预测函数 ──
def predict_herbs(symptoms: list, yolo_label: str = '', threshold: float = 0.3):
    """
    symptoms   : list[str]  患者症状列表，元素从 top_symptoms 中选取
    yolo_label : str        舌象 YOLO 标签（可为空）
    threshold  : float      概率阈值，默认 0.3
    返回: list[dict]，每项 {'herb': str, 'probability': float}，按概率降序
    """
    valid = [s for s in symptoms if s in set(_symptom_list)]
    X_sym  = _mlb_symptom.transform([valid])
    X_yolo = _mlb_yolo.transform([[yolo_label.strip()] if yolo_label.strip() else []])
    proba  = _model.predict_proba(np.hstack([X_sym, X_yolo]))[0]
    return sorted(
        [{'herb': h, 'probability': round(float(p), 4)}
         for h, p in zip(_mlb_herb.classes_, proba) if p >= threshold],
        key=lambda x: -x['probability']
    )

# ── 3. 测试 ──
result = predict_herbs(
    ['咳嗽', '喘息', '痰色白', '胸闷', '乏力'],
    'The white tongue is thick and greasy',
    threshold=0.3
)
print(f'预测中药（概率 >= 0.3），共 {len(result)} 味:')
for r in result:
    print(f"  {r['herb']:10s}  {r['probability']:.4f}")

预测中药（概率 >= 0.3），共 101 味:
  芥子          0.6799
  阿胶          0.6556
  紫石英         0.6466
  炙甘草         0.6357
  人参          0.6307
  党参          0.6175
  黄芪          0.6089
  麻黄          0.5895
  瓜蒌          0.5876
  干姜          0.5871
  川芎          0.5864
  薤白          0.5852
  陈皮          0.5827
  沉香          0.5790
  细辛          0.5769
  枳壳          0.5762
  甘草          0.5700
  山药          0.5686
  巴戟天         0.5683
  款冬花         0.5671
  黄精          0.5662
  熟地黄         0.5609
  清半夏         0.5609
  橘红          0.5598
  白术          0.5589
  白芍          0.5561
  厚朴          0.5552
  葶苈子         0.5470
  法半夏         0.5409
  半夏          0.5408
  五味子         0.5337
  穿山龙         0.5298
  茯苓          0.5294
  牡蛎          0.5252
  红景天         0.5240
  生姜          0.5237
  姜半夏         0.5233
  瓜蒌子         0.5233
  紫菀          0.5173
  紫苏子         0.5159
  百合          0.5159
  桔梗          0.5138
  当归          0.5124
  莱菔子         0.5118
  地龙          0.4974
  白果          0.4955
  山茱萸    

---
## 附：Django 集成说明

### 文件清单
```
tcm_herb_model/
├── model.pkl            # 全量数据训练的最终模型
├── mlb_symptom.pkl      # 症状编码器（固定200维）
├── mlb_yolo.pkl         # YOLO标签编码器（固定6维）
├── mlb_herb.pkl         # 中药解码器（固定150维）
├── top_symptoms.json    # 前200症状列表
├── top_herbs.json       # 前150中药列表
├── yolo_labels.json     # 6种YOLO标签列表
└── meta.json            # 元信息 + hold-out 评估指标
```

### ml_herb_service.py
```python
import pickle, json, numpy as np, os
from django.conf import settings

MODEL_DIR = os.path.join(settings.BASE_DIR, 'tcm_herb_model')
_model       = pickle.load(open(f'{MODEL_DIR}/model.pkl',       'rb'))
_mlb_symptom = pickle.load(open(f'{MODEL_DIR}/mlb_symptom.pkl', 'rb'))
_mlb_yolo    = pickle.load(open(f'{MODEL_DIR}/mlb_yolo.pkl',    'rb'))
_mlb_herb    = pickle.load(open(f'{MODEL_DIR}/mlb_herb.pkl',    'rb'))
_symptom_list = json.load(open(f'{MODEL_DIR}/top_symptoms.json', encoding='utf-8'))

def predict_herbs(symptoms, yolo_label='', threshold=0.3):
    valid = [s for s in symptoms if s in set(_symptom_list)]
    X = np.hstack([
        _mlb_symptom.transform([valid]),
        _mlb_yolo.transform([[yolo_label.strip()] if yolo_label.strip() else []])
    ])
    proba = _model.predict_proba(X)[0]
    return sorted(
        [{'herb': h, 'probability': round(float(p), 4)}
         for h, p in zip(_mlb_herb.classes_, proba) if p >= threshold],
        key=lambda x: -x['probability']
    )
```

### POST /api/predict-herbs/ 请求
```json
{"symptoms": ["咳嗽", "喘息", "痰色白"], "yolo_label": "The white tongue is thick and greasy", "threshold": 0.3}
```